# Copy of preprocessing deep aquifers

Sanitised source snapshot. See the repository README and WORKFLOW.md before running. Outputs and execution metadata are removed. Historical experiments are not an unattended pipeline or reproduced scientific result.


In [ ]:
import os
from pathlib import Path

REPO_ROOT = next(parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "prepare_wells.py").is_file())
WORKSPACE_ROOT = Path(os.environ.get("CGWB_WORKSPACE", str(REPO_ROOT / "data" / "cgwb"))).expanduser().resolve()
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE_ROOT)


In [ ]:
type_of_aquifer='deep'

# get the filtered list of mws along with their aquifer category

In [ ]:
import pandas as pd

# Path to your pickle file
file_path = f"{WORKSPACE_ROOT}/13. Checking ABY data/output/filtered_wells_based_on_nans.csv"

# Load the pickle file as a DataFrame
thresholded_mws_ids_with_aquifer_types = pd.read_csv(file_path)

# Display the first few rows
print(thresholded_mws_ids_with_aquifer_types.head())

In [ ]:
thresholded_mws_ids_with_aquifer_types

In [ ]:
thresholded_mws_ids_with_aquifer_types['id']

In [ ]:
for column in thresholded_mws_ids_with_aquifer_types.columns:
  print(column)

In [ ]:
thresholded_mws_ids_with_aquifer_types

# removing unecessary columns

In [ ]:
# List of columns to drop
columns_to_drop = [
'Sr. No.',
'State_Name_With_LGD_Code',
'District_Name_With_LGD_Code',
'Block_Name_With_LGD_Code',
'GP_Name_With_LGD_Code',
'Village',
'Site_Name',
'TYPE',
'SOURCE',
'Well_ID',
'Well_Depth (meters)',
'Aquifer',
'geometry',
'index_right',
'DN',
'consecutive_non_nan_pairs',
'index_aquifer',
'id',
'aquifer',
'aquifer0',
'aquifers',
'area_re',
'avg_mbgl',
'm2_perday',
'm3_per_day',
'mbgl',
'newcode14',
'newcode43',
'objectid',
'pa_order',
'per_cm',
'shape_Area',
'shape_Leng',
'state',
'system',
'test',
'yeild__',
'zone_m',
'longitude'
,'latitude'
]

# Create a new DataFrame without the unwanted columns
filtered_thresholded_mws_ids_with_aquifer_types = thresholded_mws_ids_with_aquifer_types.drop(columns=columns_to_drop, errors='ignore')  # errors='ignore' skips missing columns

# View the result
print(filtered_thresholded_mws_ids_with_aquifer_types.head())

In [ ]:
filtered_thresholded_mws_ids_with_aquifer_types

In [ ]:
import pandas as pd

In [ ]:
# Copy the original DataFrame
GT_mws_ids_with_aquifer_types_df = filtered_thresholded_mws_ids_with_aquifer_types.copy()

# List of years for which Pre/Post monsoon data is available
years = list(range(2015, 2023))  # 2023 excluded since we don't have Post-monsoon_2023

# Convert all Pre/Post columns to numeric (in case they're stored as strings)
for year in years:
    pre_col = f"Pre-monsoon_{year}"
    post_col = f"Post-monsoon_{year}"
    GT_mws_ids_with_aquifer_types_df[pre_col] = pd.to_numeric(GT_mws_ids_with_aquifer_types_df[pre_col], errors='coerce')
    GT_mws_ids_with_aquifer_types_df[post_col] = pd.to_numeric(GT_mws_ids_with_aquifer_types_df[post_col], errors='coerce')

# Now compute CWD_GT = Pre - Post
for year in years:
    pre_col = f"Pre-monsoon_{year}"
    post_col = f"Post-monsoon_{year}"
    cwd_gt_col = f"Pre-monsoon_{year}_CWD_GT"

    GT_mws_ids_with_aquifer_types_df[cwd_gt_col] = (
        GT_mws_ids_with_aquifer_types_df[pre_col] - GT_mws_ids_with_aquifer_types_df[post_col]
    )

In [ ]:
import pandas as pd

# Copy the original DataFrame
GT_mws_ids_with_aquifer_types_df = filtered_thresholded_mws_ids_with_aquifer_types.copy()

# List of years for which Pre/Post monsoon data is available
years = list(range(2015, 2023))  # 2023 excluded since we don't have Post-monsoon_2023

# Convert all relevant columns to numeric
for year in years:
    pre_col = f"Pre-monsoon_{year}"
    post_col = f"Post-monsoon_{year}"
    if pre_col in GT_mws_ids_with_aquifer_types_df.columns:
        GT_mws_ids_with_aquifer_types_df[pre_col] = pd.to_numeric(GT_mws_ids_with_aquifer_types_df[pre_col], errors='coerce')
    if post_col in GT_mws_ids_with_aquifer_types_df.columns:
        GT_mws_ids_with_aquifer_types_df[post_col] = pd.to_numeric(GT_mws_ids_with_aquifer_types_df[post_col], errors='coerce')

# Compute Pre-monsoon_{year}_CWD_GT = Pre - Post (same year)
for year in years:
    pre_col = f"Pre-monsoon_{year}"
    post_col = f"Post-monsoon_{year}"
    cwd_gt_col = f"Pre-monsoon_{year}_CWD_GT"

    GT_mws_ids_with_aquifer_types_df[cwd_gt_col] = (
        GT_mws_ids_with_aquifer_types_df[pre_col] - GT_mws_ids_with_aquifer_types_df[post_col]
    )

# Compute Post-monsoon_{year}_CWD_GT = Post_{year} - Pre_{year+1}
for year in years[:-1]:  # exclude 2022, as we don’t have Pre-monsoon_2023
    post_col = f"Post-monsoon_{year}"
    next_pre_col = f"Pre-monsoon_{year + 1}"
    cwd_gt_col = f"Post-monsoon_{year}_CWD_GT"

    GT_mws_ids_with_aquifer_types_df[cwd_gt_col] = (
        GT_mws_ids_with_aquifer_types_df[post_col] - GT_mws_ids_with_aquifer_types_df[next_pre_col]
    )

In [ ]:
GT_mws_ids_with_aquifer_types_df

In [ ]:
for column in GT_mws_ids_with_aquifer_types_df.columns:
  print(column)

# calculated data

In [ ]:
import pandas as pd

# Define the file path
file_path = f"{WORKSPACE_ROOT}/19. preprocessing_deep_aquifers/input/WellDepths_deep_aquifers_CSV.csv"

# Read the CSV file
deep_aquifers_calculated_df = pd.read_csv(file_path)

# Preview the first few rows
print(deep_aquifers_calculated_df.head())

In [ ]:
deep_aquifers_calculated_df

In [ ]:
for column in deep_aquifers_calculated_df.columns:
  print(column)

In [ ]:
deep_aquifers_calculated_df['2015-07-01'][10]

In [ ]:
# Filter the row by microwatershed_id
value = deep_aquifers_calculated_df.loc[
    deep_aquifers_calculated_df['microwatershed_id'] == '00080000000000006861',
    '2016-10-01'
]

# Print the value
print("Value for 2016-10-01:", value.values[0] if not value.empty else "ID not found")


In [ ]:
# Filter the row by microwatershed_id
value = deep_aquifers_calculated_df.loc[
    deep_aquifers_calculated_df['microwatershed_id'] == '0008000000000000a20a',
    '2016-10-01'
]

# Print the value
print("Value for 2016-10-01:", value.values[0] if not value.empty else "ID not found")


In [ ]:
cols_to_check = ["2020-07-01", "2020-10-01"]

for col in cols_to_check:
    if col in deep_aquifers_calculated_df.columns:
        print(f"\n--- {col} ---")
        print(deep_aquifers_calculated_df[col].apply(type).value_counts())
        print(deep_aquifers_calculated_df[col].head())

In [ ]:
import pandas as pd
import json

# Assuming the DataFrame `deep_aquifers_calculated_df` is already loaded

# Step 1: Filter out rows where weighted_avg_yeild == 0
filtered_df = deep_aquifers_calculated_df[deep_aquifers_calculated_df['weighted_avg_yeild'] != 0].copy()

# Step 2: Identify date columns
date_cols = [col for col in filtered_df.columns if col[:4].isdigit() and col[4] == '-']

# Step 3: Melt the DataFrame
melted_df = filtered_df.melt(
    id_vars=['microwatershed_id', 'weighted_avg_yeild', '.geo'],
    value_vars=date_cols,
    var_name='date',
    value_name='data'
)

# Step 4: Parse the JSON data in 'data' column
melted_df['data'] = melted_df['data'].apply(json.loads)
parsed_df = pd.json_normalize(melted_df['data'])
melted_df = pd.concat([melted_df.drop(columns=['data']), parsed_df], axis=1)

# Step 5: Extract year and season
melted_df['year'] = melted_df['date'].str[:4]
melted_df['season'] = melted_df['date'].str[5:7].map({
    '01': 1,
    '04': 2,
    '07': 3,
    '10': 4
})

# Step 6: Create final dataframe
deep_aquifer_renamed_df = melted_df.copy()

In [ ]:
deep_aquifer_renamed_df

In [ ]:
# Step 2: Group seasons 1+2 as "Pre to Post-Monsoon" and 3+4 as "Post to Pre-Monsoon"
def map_season_label(season):
    if season in [1, 2]:
        return 'Pre to Post-Monsoon'
    elif season in [3, 4]:
        return 'Post to Pre-Monsoon'

# Assign season group
deep_aquifer_renamed_df['season_label'] = deep_aquifer_renamed_df['season'].apply(map_season_label)

# Step 3: Group by year, season_label, and microwatershed_id, summing relevant metrics
agg_df = deep_aquifer_renamed_df.groupby(
    ['microwatershed_id', 'year', 'season_label', '.geo'], as_index=False
)[['DeltaG', 'ET', 'G', 'Precipitation', 'RunOff', 'WellDepth']].sum()

# Step 4: Final dataframe without date
deep_aquifer_seasonal_df = agg_df.copy()

In [ ]:
deep_aquifer_seasonal_df

#Terrain and Stream Order

In [ ]:
file_path = f'{WORKSPACE_ROOT}/19. preprocessing_deep_aquifers/input/deep_aquifer_terrain.csv'
# Read CSV into a DataFrame
terrain_filt_df = pd.read_csv(file_path)

In [ ]:
terrain_filt_df

In [ ]:
for column in terrain_filt_df.columns:
  print(column)

In [ ]:
import ast
import json
import pandas as pd

# First, create a lookup dictionary for microwatershed_id -> terrain_cluster
terrain_lookup = dict(zip(terrain_filt_df['microwatershed_id'], terrain_filt_df['terrain_cluster']))

# Seasons and years
seasons = ['Post-monsoon', 'Pre-monsoon']
years = list(range(2015, 2024))

# Create a copy of the aquifer dataframe to modify
aquifer_with_terrain_df = deep_aquifers_renamed_df.copy()

# Helper to safely parse a JSON-like string
def safe_parse_json(cell):
    if isinstance(cell, dict):
        return cell
    if isinstance(cell, str):
        try:
            parsed = ast.literal_eval(cell)
            if isinstance(parsed, dict):
                return parsed
        except:
            pass
    return None  # If it's not a valid dict or JSON string

# Add terrain_cluster to each seasonal JSON cell
for year in years:
    for season in seasons:
        col = f"{season}_{year}_calculated_CWD"
        if col in aquifer_with_terrain_df.columns:
            for idx in aquifer_with_terrain_df.index:
                cell_json = safe_parse_json(aquifer_with_terrain_df.at[idx, col])

                if cell_json is not None:
                    # Get microwatershed_id and corresponding terrain_cluster
                    mws_id = aquifer_with_terrain_df.at[idx, 'microwatershed_id']
                    terrain_cluster = terrain_lookup.get(mws_id, None)

                    # Update JSON with terrain_cluster
                    cell_json['terrain_cluster'] = terrain_cluster

                    # Save back
                    aquifer_with_terrain_df.at[idx, col] = json.dumps(cell_json)
                else:
                    # If the cell is not a valid dict/JSON, optionally skip or log
                    print(f"⚠️ Skipping row {idx} for column {col} — invalid JSON format")

print("✅ Added 'terrain_cluster' to all valid seasonal JSON fields without overwriting existing data.")


In [ ]:
# Inspect the problematic cells that are being skipped
col_name = 'Post-monsoon_2020_calculated_CWD'
problem_rows = list(range(353, 367))  # Rows 353 to 366

for idx in problem_rows:
    raw_value = aquifer_with_terrain_df.at[idx, col_name]
    print(f"Row {idx}: Type={type(raw_value)} | Value={raw_value}")

In [ ]:
aquifer_with_terrain_df

In [ ]:
aquifer_with_terrain_df['Pre-monsoon_2023_calculated_CWD'][222]

In [ ]:
import pandas as pd

# Define the file path for reading the CSV
csv_file_path = f'{WORKSPACE_ROOT}/19. preprocessing_deep_aquifers/input/deep_aquifer_SO.csv'

# Read the CSV file into a DataFrame
stream_order_continous = pd.read_csv(csv_file_path)

# Display the first few rows of the DataFrame to ensure it was read correctly
print(stream_order_continous)

In [ ]:
for column in stream_order_continous.columns:
  print(column)

In [ ]:
from tqdm import tqdm
import json

def safe_parse_json(json_str):
    if pd.isna(json_str) or not isinstance(json_str, str) or json_str.strip() == "":
        return {}
    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        print(f"⚠️ JSON parsing failed: {e}")
        return {}

# Rebuild lookup dictionary
stream_order_cols = [f'percent_{i}' for i in range(1, 12)]
stream_order_lookup = stream_order_continous.set_index('microwater')[stream_order_cols].to_dict(orient='index')

# Recopy the base DataFrame
aquifer_with_terrain_and_stream_df = aquifer_with_terrain_df.copy()

# Iterate through each year and season
for year in years:
    for season in seasons:
        col = f"{season}_{year}_calculated_CWD"
        if col not in aquifer_with_terrain_and_stream_df.columns:
            continue

        print(f"🌀 Processing: {col}")
        for idx in tqdm(aquifer_with_terrain_and_stream_df.index):
            mws_id = aquifer_with_terrain_and_stream_df.at[idx, 'microwatershed_id']
            stream_data = stream_order_lookup.get(mws_id)
            if stream_data is None:
                continue

            # Parse and preserve existing JSON
            original_json = safe_parse_json(aquifer_with_terrain_and_stream_df.at[idx, col])

            # Add stream order percentages
            for key in stream_order_cols:
                original_json[key] = stream_data.get(key, None)

            # Save back the updated JSON
            aquifer_with_terrain_and_stream_df.at[idx, col] = json.dumps(original_json)


In [ ]:
aquifer_with_terrain_and_stream_df

In [ ]:
aquifer_with_terrain_and_stream_df['Post-monsoon_2020_calculated_CWD'][322]

In [ ]:
GT_mws_ids_with_aquifer_types_df

In [ ]:
for column in GT_mws_ids_with_aquifer_types_df.columns:
  print(column)

In [ ]:
# Make sure your safe_parse_json is solid
def safe_parse_json(json_str):
    if pd.isna(json_str) or not isinstance(json_str, str) or json_str.strip() == "":
        return {}
    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        print(f"⚠️ JSON parsing failed: {e}")
        return {}

# Extract all CWD_GT columns (those ending with '_CWD_GT')
cwd_gt_cols = [col for col in GT_mws_ids_with_aquifer_types_df.columns if col.endswith('_CWD_GT')]

# Create lookup using microwatershed_id
lookup_df = GT_mws_ids_with_aquifer_types_df.set_index('microwatershed_id')

# Make a fresh copy to modify
aquifer_with_terrain_stream_and_CWD_GT_df = aquifer_with_terrain_and_stream_df.copy()

# Loop through all CWD_GT columns
for col in cwd_gt_cols:
    # Convert to seasonal JSON column name (e.g., Pre-monsoon_2015_CWD_GT → Pre-monsoon_2015_calculated_CWD)
    season_col = col.replace('_CWD_GT', '_calculated_CWD')

    if season_col not in aquifer_with_terrain_stream_and_CWD_GT_df.columns:
        continue  # Skip if that column doesn't exist

    print(f"🌀 Processing: {season_col}")

    for idx in aquifer_with_terrain_stream_and_CWD_GT_df.index:
        # Get microwatershed_id for the current row
        mws_id = aquifer_with_terrain_stream_and_CWD_GT_df.at[idx, 'microwatershed_id']

        # Retrieve values from the lookup (with safety)
        cwd_value = lookup_df.at[mws_id, col] if mws_id in lookup_df.index else None
        aquifer_category = lookup_df.at[mws_id, 'aquifer_category'] if mws_id in lookup_df.index else None
        Latitude = lookup_df.at[mws_id, 'Latitude'] if mws_id in lookup_df.index else None
        Longitude = lookup_df.at[mws_id, 'Longitude'] if mws_id in lookup_df.index else None

        # Load the existing JSON
        cell_json = safe_parse_json(aquifer_with_terrain_stream_and_CWD_GT_df.at[idx, season_col])

        # Inject the new fields
        cell_json['CWD_GT'] = cwd_value
        cell_json['aquifer_category'] = aquifer_category
        cell_json['Latitude']=Latitude
        cell_json['Longitude']=Longitude

        # Save back
        aquifer_with_terrain_stream_and_CWD_GT_df.at[idx, season_col] = json.dumps(cell_json)

print("✅ Successfully added 'CWD_GT','aquifer_category', 'Latitiude', 'Longitude' to all valid seasonal JSON cells.")

In [ ]:
aquifer_with_terrain_stream_and_CWD_GT_df

In [ ]:
aquifer_with_terrain_stream_and_CWD_GT_df['Post-monsoon_2020_calculated_CWD'][0]

In [ ]:
# Step 1: Identify seasonal JSON columns
seasonal_columns = [
    col for col in aquifer_with_terrain_stream_and_CWD_GT_df.columns
    if '_' in col and not col.endswith('.geo') and not col.startswith('system') and not col.startswith('microwatershed_id')
    and col not in ['weighted_avg_yeild', '.geo']
]

# Step 2: Check if at least one seasonal JSON has a non-null aquifer_category
def has_at_least_one_valid_aquifer_category(row):
    for col in seasonal_columns:
        json_obj = safe_parse_json(row[col])
        if json_obj.get('aquifer_category', None) is not None:
            return True  # Keep this row
    return False  # Drop this row

# Step 3: Filter and create new DataFrame
filtered_aquifer_with_terrain_stream_and_CWD_GT_df = aquifer_with_terrain_stream_and_CWD_GT_df[
    aquifer_with_terrain_stream_and_CWD_GT_df.apply(has_at_least_one_valid_aquifer_category, axis=1)
].copy()

# Optional: Reset index
filtered_aquifer_with_terrain_stream_and_CWD_GT_df.reset_index(drop=True, inplace=True)

print(f"✅ Final DataFrame created with shape: {filtered_aquifer_with_terrain_stream_and_CWD_GT_df.shape}")

In [ ]:
filtered_aquifer_with_terrain_stream_and_CWD_GT_df

In [ ]:
filtered_aquifer_with_terrain_stream_and_CWD_GT_df['Pre-monsoon_2016_calculated_CWD'][311]

In [ ]:
import json
import ast

# ✅ Reliable parser
def safe_parse_json(cell):
    if pd.isna(cell):
        return {}
    try:
        # Parse from JSON string
        if isinstance(cell, str):
            return json.loads(cell)
        elif isinstance(cell, dict):
            return cell
        else:
            return ast.literal_eval(cell)
    except Exception:
        return {}

# ✅ Fix: Append weighted_avg_yeild without wiping original content
for col in seasonal_columns:
    if col in filtered_aquifer_with_terrain_stream_and_CWD_GT_df.columns:
        for idx in filtered_aquifer_with_terrain_stream_and_CWD_GT_df.index:
            cell_value = filtered_aquifer_with_terrain_stream_and_CWD_GT_df.at[idx, col]

            json_obj = safe_parse_json(cell_value)

            # 🛑 Skip empty JSONs (you can log them if needed)
            if not isinstance(json_obj, dict) or len(json_obj) == 0:
                continue  # skip modifying this one

            # ✅ Add weighted_avg_yeild
            weighted_yield = filtered_aquifer_with_terrain_stream_and_CWD_GT_df.at[idx, 'weighted_avg_yeild']
            json_obj['weighted_avg_yeild'] = weighted_yield

            # ✅ Save back
            filtered_aquifer_with_terrain_stream_and_CWD_GT_df.at[idx, col] = json.dumps(json_obj)

In [ ]:
filtered_aquifer_with_terrain_stream_and_CWD_GT_df['Pre-monsoon_2016_calculated_CWD'][311]

In [ ]:
# ✅ Extract 'aquifer_category' from the first valid seasonal JSON
def extract_aquifer_category(row):
    for col in seasonal_columns:
        json_obj = safe_parse_json(row[col])
        ac = json_obj.get('aquifer_category', None)
        if ac is not None:
            return ac
    return None  # fallback

# ✅ Add as a new column
filtered_aquifer_with_terrain_stream_and_CWD_GT_df['aquifer_category'] = (
    filtered_aquifer_with_terrain_stream_and_CWD_GT_df.apply(extract_aquifer_category, axis=1)
)

print("✅ 'aquifer_category' column added to DataFrame (outside JSON).")


In [ ]:
filtered_aquifer_with_terrain_stream_and_CWD_GT_df

In [ ]:
# Count rows by aquifer category
category_counts = filtered_aquifer_with_terrain_stream_and_CWD_GT_df['aquifer_category'].value_counts()

# Total from categories
total_from_categories = category_counts.sum()

# Actual total rows in the DataFrame
total_rows = len(filtered_aquifer_with_terrain_stream_and_CWD_GT_df)

# Display breakdown
print("✅ Aquifer Category Counts:")
for category, count in category_counts.items():
    print(f"- {category}: {count}")

print(f"\n🔢 Total from categories: {total_from_categories}")
print(f"📋 Total rows in DataFrame: {total_rows}")

# ✅ Check consistency
if total_from_categories == total_rows:
    print("\n✅ Consistency Check Passed: Totals match! 🎯")
else:
    print("\n❌ Warning: Totals do NOT match! 🚨")

In [ ]:
flattened_rows = []
non_seasonal_columns = ['microwatershed_id', 'weighted_avg_yeild', 'aquifer_category', '.geo']

for idx, row in filtered_aquifer_with_terrain_stream_and_CWD_GT_df.iterrows():
    base_info = row[non_seasonal_columns].to_dict()

    for col in seasonal_columns:
        json_data = safe_parse_json(row[col])

        if not json_data or not isinstance(json_data, dict):
            continue

        # Example col: 'Pre-monsoon_2015_calculated_CWD'
        if col.startswith("Pre-monsoon") or col.startswith("Post-monsoon"):
            try:
                season_part, year_part, *_ = col.split("_")
                season = season_part
                year = int(year_part)
            except:
                continue  # Skip if parsing fails
        else:
            continue

        flattened_row = {
            **base_info,
            'season': season,
            'year': year,
            **json_data
        }

        flattened_rows.append(flattened_row)

# Final unwrapped DataFrame
unwrapped_df = pd.DataFrame(flattened_rows)

print(f"✅ Unwrapped DataFrame created with shape: {unwrapped_df.shape}")
unwrapped_df.head()


In [ ]:
unwrapped_df

In [ ]:
for column in unwrapped_df.columns:
  print(column)

In [ ]:
# Rename the column
unwrapped_df.rename(columns={'WellDepth': 'CWD_Calculated'}, inplace=True)

# Confirm the change
print("✅ Renamed 'WellDepth' to 'CWD_Calculated'")

In [ ]:
filtered_unwrapped_df = unwrapped_df.dropna().copy()
print(f"✅ Filtered unwrapped DataFrame shape (no missing values in any column): {filtered_unwrapped_df.shape}")

# training prep

In [ ]:
# Step 1: Map season to ordinal
season_order = {
    'Post-monsoon': 1,
    'Pre-monsoon': 2,
}
filtered_unwrapped_df['season_ordinal'] = filtered_unwrapped_df['season'].map(season_order)

# Step 2: Define columns
drop_cols = ['.geo', 'season']
categorical_cols = ['aquifer_category', 'terrain_cluster']
numerical_cols = [
    'DeltaG', 'ET', 'G', 'Precipitation', 'RunOff', 'CWD_Calculated',
    'weighted_avg_yeild', 'season_ordinal', 'year','Latitude',
    'Longitude' # ✅ year added
] + [f'percent_{i}' for i in range(1, 12)]

# Step 3: Final feature list and modeling dataframe
feature_cols = categorical_cols + numerical_cols
model_df = filtered_unwrapped_df.drop(columns=drop_cols).copy()

print("✅ Final DataFrame for modeling with 'year' included:")
print(model_df[feature_cols + ['CWD_GT']].head())

In [ ]:
model_df

In [ ]:
model_df['aquifer_category'].unique()

In [ ]:
model_df.columns

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

# 1. Load India boundary GeoJSON
india_url = f"{WORKSPACE_ROOT}/19. preprocessing_deep_aquifers/input/Indian_states_boundary.geojson"
india = gpd.read_file(india_url)

# 2. Create GeoDataFrame for points
gdf = gpd.GeoDataFrame(
    model_df.dropna(subset=['Latitude', 'Longitude', 'aquifer_category']),
    geometry=gpd.points_from_xy(model_df['Longitude'], model_df['Latitude']),
    crs='EPSG:4326'
)

# 3. Define custom colors
category_colors = {
    'Alluvial': '#FFDBAC',
    'Himalayan Systems': 'peru',
    'Volcanic Systems': 'mediumseagreen',
    'Sedimentary Hard Rock': 'dodgerblue',
    'Sedimentary Soft Rock': 'skyblue',
    'Crystalline Systems': 'lightpink'
}

# 4. Plot
fig, ax = plt.subplots(figsize=(10, 10))
india.plot(ax=ax, color='white', edgecolor='black')

# Prepare legend handles manually for control
from matplotlib.lines import Line2D
legend_elements = []

for category, color in category_colors.items():
    subset = gdf[gdf['aquifer_category'] == category]
    if not subset.empty and subset.geometry.is_valid.all():
        subset.plot(ax=ax, markersize=10, color=color, alpha=0.6)
        legend_elements.append(
            Line2D([0], [0], marker='o', color='w', label=category,
                   markerfacecolor=color, markersize=10)  # Increase markersize here
        )

# Custom legend with larger marker size and font
ax.legend(
    handles=legend_elements,
    title="Aquifer Category",
    title_fontsize='large',
    fontsize='large',         # Bigger font size
    loc='upper right'
)

plt.title("Microwatershed Locations by Aquifer Category")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

In [ ]:
# Once the model_df is loaded, proceed with vertical yellow boxplots
# Ensure to remove excluded columns
exclude_cols = ['microwatershed_id', 'aquifer_category', 'season_ordinal']
numeric_cols = [col for col in model_df.columns if col not in exclude_cols]

# Plot each column as a vertical boxplot with yellow color
for col in numeric_cols:
    plt.figure(figsize=(2, 6))
    sns.boxplot(y=model_df[col], color='gold')  # gold is a nice yellow shade
    plt.title(f" Boxplot for {col}")
    plt.tight_layout()
    plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

# Step 1: Get unique microwatershed IDs
unique_ids = model_df['microwatershed_id'].unique()

# Step 2: Split IDs into train/test
train_ids, test_ids = train_test_split(unique_ids, test_size=0.2, random_state=42)

# Step 3: Subset the model_df based on ID membership
train_df = model_df[model_df['microwatershed_id'].isin(train_ids)].copy()
test_df = model_df[model_df['microwatershed_id'].isin(test_ids)].copy()

print(f"✅ Train shape: {train_df.shape}, Test shape: {test_df.shape}")

In [ ]:
import pandas as pd
import statsmodels.api as sm

In [ ]:
# Step 4: Encode categorical features in train/test separately (to avoid leakage)
train_encoded = pd.get_dummies(train_df, columns=['aquifer_category', 'terrain_cluster'], drop_first=True)
test_encoded = pd.get_dummies(test_df, columns=['aquifer_category', 'terrain_cluster'], drop_first=True)

# Align columns to handle missing dummies in test
train_encoded, test_encoded = train_encoded.align(test_encoded, join='left', axis=1, fill_value=0)

# Step 5: Prepare X and y
X_train = train_encoded.drop(columns=['CWD_GT', '.geo', 'season', 'microwatershed_id'], errors='ignore')
y_train = train_encoded['CWD_GT']

X_test = test_encoded.drop(columns=['CWD_GT','.geo', 'season', 'microwatershed_id'], errors='ignore')
y_test = test_encoded['CWD_GT']

# Ensure everything in X_train and y_train is numeric
X_train = X_train.apply(pd.to_numeric, errors='coerce')
y_train = pd.to_numeric(y_train, errors='coerce')

X_test = X_test.apply(pd.to_numeric, errors='coerce')
y_test = pd.to_numeric(y_test, errors='coerce')

# Drop any rows with NaNs (if any were introduced)
X_train = X_train.dropna()
y_train = y_train.loc[X_train.index]  # match indices

X_test = X_test.dropna()
y_test = y_test.loc[X_test.index]

# Convert all boolean columns to int (0/1)
X_train = X_train.astype({col: int for col in X_train.select_dtypes('bool').columns})
X_test = X_test.astype({col: int for col in X_test.select_dtypes('bool').columns})

# Step 6: Add constant term
X_train = sm.add_constant(X_train)
X_test = sm.add_constant(X_test)

In [ ]:
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

# Step 4: Encode categorical features in train/test separately (to avoid leakage)
train_encoded = pd.get_dummies(train_df, columns=['aquifer_category', 'terrain_cluster'], drop_first=True)
test_encoded = pd.get_dummies(test_df, columns=['aquifer_category', 'terrain_cluster'], drop_first=True)

# Align columns to handle missing dummies in test
train_encoded, test_encoded = train_encoded.align(test_encoded, join='left', axis=1, fill_value=0)

# Step 5: Prepare X and y
X_train = train_encoded.drop(columns=['CWD_GT', '.geo', 'season', 'microwatershed_id'], errors='ignore')
y_train = train_encoded['CWD_GT']

X_test = test_encoded.drop(columns=['CWD_GT','.geo', 'season', 'microwatershed_id'], errors='ignore')
y_test = test_encoded['CWD_GT']

# Ensure everything in X_train and y_train is numeric
X_train = X_train.apply(pd.to_numeric, errors='coerce')
y_train = pd.to_numeric(y_train, errors='coerce')

X_test = X_test.apply(pd.to_numeric, errors='coerce')
y_test = pd.to_numeric(y_test, errors='coerce')

# Drop any rows with NaNs (if any were introduced)
X_train = X_train.dropna()
y_train = y_train.loc[X_train.index]

X_test = X_test.dropna()
y_test = y_test.loc[X_test.index]

# Convert boolean columns to integers
X_train = X_train.astype({col: int for col in X_train.select_dtypes('bool').columns})
X_test = X_test.astype({col: int for col in X_test.select_dtypes('bool').columns})

# ✅ Step 6: Normalize using StandardScaler (zero mean, unit variance)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

# ✅ Step 7: Add constant term for OLS (after scaling)
X_train_scaled = sm.add_constant(X_train_scaled)
X_test_scaled = sm.add_constant(X_test_scaled)


In [ ]:
print("🔎 X_train dtypes:")
print(X_train.dtypes)

print("\n🔎 y_train dtype:")
print(y_train.dtype)

In [ ]:
print("🔎 X_train dtypes:")
print(X_test.dtypes)

print("\n🔎 y_train dtype:")
print(y_test.dtype)

In [ ]:
# Fit OLS
ols_model = sm.OLS(y_train, sm.add_constant(X_train)).fit()

# Predict on test set
y_pred = ols_model.predict(sm.add_constant(X_test))

# Summary
print(ols_model.summary())

In [ ]:
# Get the summary as a DataFrame
summary_df = ols_model.summary2().tables[1]

# Filter by p-value
significant_vars = summary_df[summary_df['P>|t|'] < 0.05]

print("✅ Statistically significant covariates (p < 0.05):")
print(significant_vars[['Coef.', 'P>|t|']])

In [ ]:
# Get the summary table as a DataFrame
coef_df = ols_model.summary2().tables[1]

# Option 2: Sort by actual signed coefficient (positive to negative)
sorted_by_value = coef_df['Coef.'].sort_values(ascending=False)

print("\n📈 Coefficients sorted by value (from most positive to most negative):")
print(sorted_by_value)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Predict for train and test
y_train_pred = ols_model.predict(sm.add_constant(X_train))
y_test_pred = ols_model.predict(sm.add_constant(X_test))

# Plot
plt.figure(figsize=(12, 5))

# --- Train plot ---
plt.subplot(1, 2, 1)
sns.scatterplot(x=y_train, y=y_train_pred, alpha=0.6)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')
plt.xlabel("Actual CWD_GT (Train)")
plt.ylabel("Predicted CWD_GT")
plt.title("Train Set: Predicted vs Actual")

# --- Test plot ---
plt.subplot(1, 2, 2)
sns.scatterplot(x=y_test, y=y_test_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel("Actual CWD_GT (Test)")
plt.ylabel("Predicted CWD_GT")
plt.title("Test Set: Predicted vs Actual")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

# Step 1: Get all unique microwatershed IDs
unique_ids = model_df['microwatershed_id'].unique()

# Step 2: First split → train (80%) and temp (20%)
train_ids, temp_ids = train_test_split(unique_ids, test_size=0.2, random_state=42)

# Step 3: Second split → temp into test (10%) and validation (10%)
test_ids, val_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)

# Step 4: Create the DataFrames
train_df = model_df[model_df['microwatershed_id'].isin(train_ids)].copy()
test_df = model_df[model_df['microwatershed_id'].isin(test_ids)].copy()
val_df = model_df[model_df['microwatershed_id'].isin(val_ids)].copy()

print(f"✅ Shapes - Train: {train_df.shape}, Test: {test_df.shape}, Validation: {val_df.shape}")

In [ ]:
# One-hot encode categorical features
train_encoded = pd.get_dummies(train_df, columns=['aquifer_category', 'terrain_cluster'], drop_first=True)
test_encoded = pd.get_dummies(test_df, columns=['aquifer_category', 'terrain_cluster'], drop_first=True)
val_encoded = pd.get_dummies(val_df, columns=['aquifer_category', 'terrain_cluster'], drop_first=True)

# Align columns across all sets
train_encoded, test_encoded = train_encoded.align(test_encoded, join='left', axis=1, fill_value=0)
train_encoded, val_encoded = train_encoded.align(val_encoded, join='left', axis=1, fill_value=0)

# Extract X and y
drop_cols = ['CWD_GT', 'system:index', '.geo', 'season', 'microwatershed_id']

X_train = train_encoded.drop(columns=drop_cols, errors='ignore').astype(float)
y_train = train_encoded['CWD_GT']

X_test = test_encoded.drop(columns=drop_cols, errors='ignore').astype(float)
y_test = test_encoded['CWD_GT']

X_val = val_encoded.drop(columns=drop_cols, errors='ignore').astype(float)
y_val = val_encoded['CWD_GT']

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import ParameterGrid

# Assume X_train, y_train, X_val, y_val, X_test, y_test are already defined

# Define models and their hyperparameter grids
model_grid = {
    'LinearRegression': {
        'model': LinearRegression(),
        'params': [{}]
    },
    'Ridge': {
        'model': Ridge(),
        'params': {'alpha': [0.01, 0.1, 1, 10, 100]}
    },
    'Lasso': {
        'model': Lasso(max_iter=10000),
        'params': {'alpha': [0.001, 0.01, 0.1, 1, 10]}
    },
    'ElasticNet': {
        'model': ElasticNet(max_iter=10000),
        'params': {'alpha': [0.001, 0.01, 0.1], 'l1_ratio': [0.2, 0.5, 0.8]}
    },
    'DecisionTree': {
        'model': DecisionTreeRegressor(random_state=42),
        'params': {'max_depth': [5, 10, 20, None]}
    },
    'RandomForest': {
        'model': RandomForestRegressor(random_state=42),
        'params': {'n_estimators': [50, 100], 'max_depth': [10, 20, None]}
    },
    'ExtraTrees': {
        'model': ExtraTreesRegressor(random_state=42),
        'params': {'n_estimators': [50, 100], 'max_depth': [10, 20, None]}
    },
    'GradientBoosting': {
        'model': GradientBoostingRegressor(random_state=42),
        'params': {'n_estimators': [50, 100], 'learning_rate': [0.01, 0.1], 'max_depth': [3, 5]}
    },
    'KNN': {
        'model': KNeighborsRegressor(),
        'params': {'n_neighbors': [3, 5, 7]}
    },
    'SVR': {
        'model': SVR(),
        'params': {'C': [0.1, 1, 10], 'epsilon': [0.01, 0.1]}
    },
    'MLP': {
        'model': MLPRegressor(max_iter=1000, random_state=42),
        'params': {'hidden_layer_sizes': [(50,), (100,), (100, 50)], 'alpha': [0.0001, 0.001]}
    }
}

results = []

# Evaluate all models
for name, config in model_grid.items():
    best_mae = float('inf')
    best_model = None
    best_params = None

    for params in ParameterGrid(config['params']):
        model = config['model'].set_params(**params)
        model.fit(X_train, y_train)
        val_pred = model.predict(X_val)
        val_mae = mean_absolute_error(y_val, val_pred)

        if val_mae < best_mae:
            best_mae = val_mae
            best_model = model
            best_params = params

    # Retrain on train + val
    X_combined = np.vstack([X_train, X_val])
    y_combined = np.hstack([y_train, y_val])
    final_model = config['model'].set_params(**best_params)
    final_model.fit(X_combined, y_combined)

    # Predict on train and test
    train_pred = final_model.predict(X_train)
    test_pred = final_model.predict(X_test)

    # MAE
    train_mae = mean_absolute_error(y_train, train_pred)
    test_mae = mean_absolute_error(y_test, test_pred)

    results.append({
        'Model': name,
        'Best Params': best_params,
        'Train MAE': train_mae,
        'Val MAE': best_mae,
        'Test MAE': test_mae
    })

    # Plot
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    sns.scatterplot(x=y_train, y=train_pred, alpha=0.5)
    plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')
    plt.xlabel("Actual Train")
    plt.ylabel("Predicted Train")
    plt.title(f"{name} - Train")

    plt.subplot(1, 2, 2)
    sns.scatterplot(x=y_test, y=test_pred, alpha=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    plt.xlabel("Actual Test")
    plt.ylabel("Predicted Test")
    plt.title(f"{name} - Test")

    plt.suptitle(f"{name} — MAE (Train: {train_mae:.3f}, Val: {best_mae:.3f}, Test: {test_mae:.3f})")
    plt.tight_layout()
    plt.show()

#import ace_tools as tools; tools.display_dataframe_to_user(name="Model Comparison Results", dataframe=pd.DataFrame(results))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

import pickle

In [ ]:
# Define aquifer categories from the loaded DataFrame
aquifer_categories = model_df['aquifer_category'].dropna().unique()
aquifer_categories

In [ ]:
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_percentage_error, r2_score
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt

ols_results = []

# Mapping from ordinal to season names
season_map = {
  1: 'Post to Pre Monsoon (Jan–June)', 2: 'Pre to Post Monsoon (July–Dec)'
}

# Loop over aquifer category and season
for category in model_df['aquifer_category'].unique():
    for season_ordinal, season_name in season_map.items():

        df_subset = model_df[(model_df['aquifer_category'] == category) &
                     (model_df['season_ordinal'] == season_ordinal)].copy()

        # Drop rows with 0 in any of the key continuous variables
        non_onehot_cols = ['DeltaG', 'ET', 'Precipitation', 'RunOff', 'CWD_Calculated', 'weighted_avg_yeild']
        df_subset = df_subset[(df_subset[non_onehot_cols] != 0).all(axis=1)].copy()


        if df_subset.empty:
            print(f"Skipping {category} - {season_name} (no data)")
            continue

        # Split based on unique microwatershed IDs
        unique_ids = df_subset['microwatershed_id'].unique()
        if len(unique_ids) < 5:
            print(f"Not enough data for {category} - {season_name}")
            continue

        train_ids, temp_ids = train_test_split(unique_ids, test_size=0.2, random_state=42)
        test_ids, val_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)

        train_df = df_subset[df_subset['microwatershed_id'].isin(train_ids)].copy()
        test_df = df_subset[df_subset['microwatershed_id'].isin(test_ids)].copy()

        # One-hot encoding
        train_encoded = pd.get_dummies(train_df, columns=['aquifer_category', 'terrain_cluster'], drop_first=True)
        test_encoded = pd.get_dummies(test_df, columns=['aquifer_category', 'terrain_cluster'], drop_first=True)
        train_encoded, test_encoded = train_encoded.align(test_encoded, join='left', axis=1, fill_value=0)

        # Drop non-numeric or non-feature columns
        drop_cols = ['CWD_GT', 'system:index', '.geo', 'season', 'microwatershed_id', 'G','DeltaG', 'season_label']
        X_train = train_encoded.drop(columns=drop_cols, errors='ignore').astype(float)
        y_train = train_encoded['CWD_GT']
        X_test = test_encoded.drop(columns=drop_cols, errors='ignore').astype(float)
        y_test = test_encoded['CWD_GT']

        # Add constant term for OLS
        X_train_ols = sm.add_constant(X_train, has_constant='add')
        X_test_ols = sm.add_constant(X_test, has_constant='add')

        # Align test columns to match train
        X_test_ols = X_test_ols[X_train_ols.columns]

        # Fit OLS model
        ols_model = sm.OLS(y_train, X_train_ols).fit()

        # Predict
        test_pred_ols = ols_model.predict(X_test_ols)

        # Metrics
        test_mape_ols = mean_absolute_percentage_error(y_test, test_pred_ols)
        test_r2_ols = r2_score(y_test, test_pred_ols)
        train_pred_ols = ols_model.predict(X_train_ols)
        train_mape_ols = mean_absolute_percentage_error(y_train, train_pred_ols)
        train_r2_ols = r2_score(y_train, train_pred_ols)

        print(f"\nOLS Summary for Aquifer: {category} | Season: {season_name}")
        print(ols_model.summary())

        # Store results
        ols_results.append({
            'Aquifer Category': category,
            'Season': season_name,
            'Train MAPE': train_mape_ols,
            'Test MAPE': test_mape_ols,
            'Train R2': train_r2_ols,
            'Test R2': test_r2_ols
        })

        # Plot
        plt.figure(figsize=(6, 5))
        sns.scatterplot(x=y_test, y=test_pred_ols, alpha=0.6)
        plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
        plt.title(f"{category} | {season_name} - OLS Test Predictions")
        plt.xlabel("Actual")
        plt.ylabel("Predicted")
        plt.tight_layout()
        plt.show()

# Convert results to DataFrame
ols_results_df = pd.DataFrame(ols_results)


In [ ]:
from sklearn.model_selection import ParameterGrid

# Define models and their hyperparameter grids
model_grid = {
    'LinearRegression': {
        'model': LinearRegression(),
        'params': [{}]
    },
    'Ridge': {
        'model': Ridge(),
        'params': {'alpha': [0.01, 0.1, 1, 10]}
    },
    'Lasso': {
        'model': Lasso(max_iter=10000),
        'params': {'alpha': [0.001, 0.01, 0.1, 1]}
    },
    'ElasticNet': {
        'model': ElasticNet(max_iter=10000),
        'params': {'alpha': [0.001, 0.01, 0.1], 'l1_ratio': [0.2, 0.5, 0.8]}
    },
    'DecisionTree': {
        'model': DecisionTreeRegressor(random_state=42),
        'params': {'max_depth': [5, 10, None]}
    },
    'RandomForest': {
        'model': RandomForestRegressor(random_state=42),
        'params': {'n_estimators': [50, 100], 'max_depth': [10, None]}
    },
    'GradientBoosting': {
        'model': GradientBoostingRegressor(random_state=42),
        'params': {'n_estimators': [50, 100], 'learning_rate': [0.01, 0.1]}
    },
    'KNN': {
        'model': KNeighborsRegressor(),
        'params': {'n_neighbors': [3, 5, 7]}
    },
    'SVR': {
        'model': SVR(),
        'params': {'C': [0.1, 1, 10], 'epsilon': [0.01, 0.1]}
    }
}

all_aquifer_results = {}
cols_to_check = [
    'weighted_avg_yeild',
    'year',
    'DeltaG',
    'ET',
    'G',
    'Precipitation',
    'CWD_Calculated',
    'Latitude',
    'Longitude',
    'season_ordinal'
]


for category in aquifer_categories:
    df_aquifer = model_df[model_df['aquifer_category'] == category].copy()

    # Remove rows with any zero entry in any column
    df_aquifer = df_aquifer[~(df_aquifer[cols_to_check] == 0).any(axis=1)]

    unique_ids = df_aquifer['microwatershed_id'].unique()
    train_ids, temp_ids = train_test_split(unique_ids, test_size=0.2, random_state=42)
    test_ids, val_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)

    train_df = df_aquifer[df_aquifer['microwatershed_id'].isin(train_ids)].copy()
    test_df = df_aquifer[df_aquifer['microwatershed_id'].isin(test_ids)].copy()
    val_df = df_aquifer[df_aquifer['microwatershed_id'].isin(val_ids)].copy()

    train_encoded = pd.get_dummies(train_df, columns=['aquifer_category', 'terrain_cluster'], drop_first=True)
    test_encoded = pd.get_dummies(test_df, columns=['aquifer_category', 'terrain_cluster'], drop_first=True)
    val_encoded = pd.get_dummies(val_df, columns=['aquifer_category', 'terrain_cluster'], drop_first=True)

    train_encoded, test_encoded = train_encoded.align(test_encoded, join='left', axis=1, fill_value=0)
    train_encoded, val_encoded = train_encoded.align(val_encoded, join='left', axis=1, fill_value=0)

    drop_cols = [
        'CWD_GT', 'system:index', '.geo', 'season',
        'microwatershed_id', 'DeltaG', 'season_label'
    ]
    X_train = train_encoded.drop(columns=drop_cols, errors='ignore').astype(float)
    y_train = train_encoded['CWD_GT']
    X_test = test_encoded.drop(columns=drop_cols, errors='ignore').astype(float)
    y_test = test_encoded['CWD_GT']
    X_val = val_encoded.drop(columns=drop_cols, errors='ignore').astype(float)
    y_val = val_encoded['CWD_GT']

    category_results = []

    for name, config in model_grid.items():
        best_mae = float('inf')
        best_model = None
        best_params = None

        for params in ParameterGrid(config['params']):
            model = config['model'].set_params(**params)
            model.fit(X_train, y_train)
            val_pred = model.predict(X_val)
            val_mae = mean_absolute_error(y_val, val_pred)

            if val_mae < best_mae:
                best_mae = val_mae
                best_model = model
                best_params = params

        X_combined = np.vstack([X_train, X_val])
        y_combined = np.hstack([y_train, y_val])
        final_model = config['model'].set_params(**best_params)
        final_model.fit(X_combined, y_combined)

        train_pred = final_model.predict(X_train)
        test_pred = final_model.predict(X_test)

        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)
        train_r2 = r2_score(y_train, train_pred)
        test_r2 = r2_score(y_test, test_pred)

        category_results.append({
            'Model': name,
            'Best Params': best_params,
            'Train MAE': train_mae,
            'Val MAE': best_mae,
            'Test MAE': test_mae,
            'Train R2': train_r2,
            'Test R2': test_r2
        })

        # Plot scatter
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1)
        sns.scatterplot(x=y_train, y=train_pred, alpha=0.6)
        plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')
        plt.title(f"{category} - {name} Train")
        plt.xlabel("Actual")
        plt.ylabel("Predicted")

        plt.subplot(1, 2, 2)
        sns.scatterplot(x=y_test, y=test_pred, alpha=0.6)
        plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
        plt.title(f"{category} - {name} Test")
        plt.xlabel("Actual")
        plt.ylabel("Predicted")

        plt.suptitle(f"{category} | {name} | Train MAE: {train_mae:.3f}, Test MAE: {test_mae:.3f}")
        plt.tight_layout()
        plt.show()

    all_aquifer_results[category] = pd.DataFrame(category_results)

# Combine and display results
combined_results_df = pd.concat(all_aquifer_results, names=["Aquifer Category", "Model Index"]).reset_index()
#tools.display_dataframe_to_user(name="All Aquifer Categories - Model Comparison", dataframe=combined_results_df)


In [ ]:
season_map = {1: "Pre-Monsoon", 2: "Post-Monsoon"}
cols_to_check = [
    'weighted_avg_yeild', 'year', 'DeltaG', 'ET', 'G',
    'Precipitation', 'CWD_Calculated', 'Latitude', 'Longitude', 'season_ordinal'
]

all_aquifer_results = {}

for category in aquifer_categories:
    for season_code, season_name in season_map.items():
        df_filtered = model_df[
            (model_df['aquifer_category'] == category) &
            (model_df['season_ordinal'] == season_code)
        ].copy()

        # Drop rows with 0 in critical columns
        df_filtered = df_filtered[~(df_filtered[cols_to_check] == 0).any(axis=1)]

        if df_filtered.empty:
            print(f"Skipping {category} - {season_name} (no data after filtering)")
            continue

        # Train/Val/Test Split by microwatershed_id
        unique_ids = df_filtered['microwatershed_id'].unique()
        if len(unique_ids) < 5:
            print(f"Skipping {category} - {season_name} (not enough IDs)")
            continue

        train_ids, temp_ids = train_test_split(unique_ids, test_size=0.2, random_state=42)
        test_ids, val_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)

        train_df = df_filtered[df_filtered['microwatershed_id'].isin(train_ids)].copy()
        test_df = df_filtered[df_filtered['microwatershed_id'].isin(test_ids)].copy()
        val_df = df_filtered[df_filtered['microwatershed_id'].isin(val_ids)].copy()

        # One-hot encoding
        train_encoded = pd.get_dummies(train_df, columns=['aquifer_category', 'terrain_cluster'], drop_first=True)
        test_encoded = pd.get_dummies(test_df, columns=['aquifer_category', 'terrain_cluster'], drop_first=True)
        val_encoded = pd.get_dummies(val_df, columns=['aquifer_category', 'terrain_cluster'], drop_first=True)

        # Align columns
        train_encoded, test_encoded = train_encoded.align(test_encoded, join='left', axis=1, fill_value=0)
        train_encoded, val_encoded = train_encoded.align(val_encoded, join='left', axis=1, fill_value=0)

        drop_cols = ['CWD_GT', 'system:index', '.geo', 'season', 'microwatershed_id', 'season_label']
        X_train = train_encoded.drop(columns=drop_cols, errors='ignore').astype(float)
        y_train = train_encoded['CWD_GT']
        X_test = test_encoded.drop(columns=drop_cols, errors='ignore').astype(float)
        y_test = test_encoded['CWD_GT']
        X_val = val_encoded.drop(columns=drop_cols, errors='ignore').astype(float)
        y_val = val_encoded['CWD_GT']

        category_results = []

        for name, config in model_grid.items():
            best_mae = float('inf')
            best_model = None
            best_params = None

            for params in ParameterGrid(config['params']):
                model = config['model'].set_params(**params)
                model.fit(X_train, y_train)
                val_pred = model.predict(X_val)
                val_mae = mean_absolute_error(y_val, val_pred)

                if val_mae < best_mae:
                    best_mae = val_mae
                    best_model = model
                    best_params = params

            X_combined = np.vstack([X_train, X_val])
            y_combined = np.hstack([y_train, y_val])
            final_model = config['model'].set_params(**best_params)
            final_model.fit(X_combined, y_combined)

            train_pred = final_model.predict(X_train)
            test_pred = final_model.predict(X_test)

            train_mae = mean_absolute_error(y_train, train_pred)
            test_mae = mean_absolute_error(y_test, test_pred)
            train_r2 = r2_score(y_train, train_pred)
            test_r2 = r2_score(y_test, test_pred)

            category_results.append({
                'Model': name,
                'Season': season_name,
                'Best Params': best_params,
                'Train MAE': train_mae,
                'Val MAE': best_mae,
                'Test MAE': test_mae,
                'Train R2': train_r2,
                'Test R2': test_r2
            })

            # Plotting
            plt.figure(figsize=(12, 5))
            plt.subplot(1, 2, 1)
            sns.scatterplot(x=y_train, y=train_pred, alpha=0.6)
            plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')
            plt.title(f"{category} - {season_name} - {name} Train")
            plt.xlabel("Actual")
            plt.ylabel("Predicted")

            plt.subplot(1, 2, 2)
            sns.scatterplot(x=y_test, y=test_pred, alpha=0.6)
            plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
            plt.title(f"{category} - {season_name} - {name} Test")
            plt.xlabel("Actual")
            plt.ylabel("Predicted")

            plt.suptitle(f"{category} | {season_name} | {name} | Train MAE: {train_mae:.3f}, Test MAE: {test_mae:.3f}")
            plt.tight_layout()
            plt.show()

        all_aquifer_results[(category, season_name)] = pd.DataFrame(category_results)

# Combine all results
combined_results_df = pd.concat(all_aquifer_results, names=["Aquifer Category", "Season", "Model Index"]).reset_index()


In [ ]:
# Dictionary to store train and test dataframes
aquifer_data_splits = {}

for category in aquifer_categories:

  # Inside the loop (after preparing train_df and test_df)
  aquifer_data_splits[category] = {
      'train_df': train_df.copy(),
      'test_df': test_df.copy()
  }


In [ ]:
aquifer_data_splits

In [ ]:
import os

In [ ]:
base_path = f"{WORKSPACE_ROOT}/19. preprocessing_deep_aquifers/output"

# Ensure the directory exists
# os.makedirs(base_path, exist_ok=True)

In [ ]:
for category, splits in aquifer_data_splits.items():
    train_file = os.path.join(base_path, f"train_data_{category}.csv")
    test_file = os.path.join(base_path, f"test_data_{category}.csv")

    splits['train_df'].to_csv(train_file, index=False)
    splits['test_df'].to_csv(test_file, index=False)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Columns you want to visualize
cols_to_plot = [
    'weighted_avg_yeild',
    'year',
    'DeltaG',
    'ET',
    'G',
    'Precipitation',
    'CWD_Calculated',
    'Latitude',
    'Longitude',
    'season_ordinal',
    'RunOff'
]

season_map = {
  1: 'Post to Pre Monsoon (Jan–June)', 2: 'Pre to Post Monsoon (July–Dec)'
}
model_df['season_label'] = model_df['season_ordinal'].map(season_map)

# Create boxplots for each aquifer type
for category in model_df['aquifer_category'].unique():
    df_subset = model_df[model_df['aquifer_category'] == category]

    for col in cols_to_plot:
        if col not in df_subset.columns:
            print(f"Skipping '{col}' – not found in DataFrame.")
            continue

        plt.figure(figsize=(8, 5))
        sns.boxplot(data=df_subset, x='season_label', y=col)
        plt.title(f'Aquifer: {category} | {col} by Season')
        plt.xlabel('Season')
        plt.ylabel(col)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# List of columns to plot
cols_to_plot = [
    'Precipitation',
    'RunOff',
    'ET',
    'CWD_Calculated',
    'CWD_GT',
    'weighted_avg_yeild'
]

# Mapping for season labels
season_map = {
  1: 'Post to Pre Monsoon (Jan–June)', 2: 'Pre to Post Monsoon (July–Dec)'
}
model_df['season_label'] = model_df['season_ordinal'].map(season_map)

# Unique values
aquifer_categories = model_df['aquifer_category'].unique()
seasons = model_df['season_label'].dropna().unique()

# Plot side-by-side boxplots in grid (2 rows x 5 cols)
n_cols = 5
n_rows = (len(cols_to_plot) + n_cols - 1) // n_cols  # Round up
figsize = (n_cols * 5, n_rows * 5)

for category in aquifer_categories:
    for season in seasons:
        df_subset = model_df[(model_df['aquifer_category'] == category) & (model_df['season_label'] == season)]

        if df_subset.empty:
            print(f"Skipping {category} - {season} (no data)")
            continue

        fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
        axes = axes.flatten()

        for i, col in enumerate(cols_to_plot):
            if col not in df_subset.columns:
                print(f"Column '{col}' not found in DataFrame.")
                continue

            sns.boxplot(data=df_subset, y=col, ax=axes[i])
            axes[i].set_title(col, fontsize=14)
            axes[i].tick_params(axis='y', labelsize=12)

        # Hide any unused subplots
        for j in range(len(cols_to_plot), len(axes)):
            fig.delaxes(axes[j])

        plt.suptitle(f"Aquifer: {category} | Season: {season}", fontsize=18)
        plt.tight_layout(rect=[0, 0, 1, 0.95])  # Leave space for title
        plt.show()


In [ ]:
india_url = f"{WORKSPACE_ROOT}/19. preprocessing_deep_aquifers/input/Indian_states_boundary.geojson"
india_state_boundary_map = gpd.read_file(india_url)

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point

# 1. Read India state boundary
india_url = f"{WORKSPACE_ROOT}/19. preprocessing_deep_aquifers/input/Indian_states_boundary.geojson"
india_state_boundary_map = gpd.read_file(india_url)

# 2. Convert model_df to GeoDataFrame if not already
if 'geometry' not in model_df.columns:
    # Assuming your model_df has 'longitude' and 'latitude' columns
    model_gdf = gpd.GeoDataFrame(
        model_df,
        geometry=gpd.points_from_xy(model_df.Longitude, model_df.Latitude),
        crs='EPSG:4326'
    )
else:
    model_gdf = gpd.GeoDataFrame(model_df, geometry='geometry')
    if model_gdf.crs is None:
        model_gdf.set_crs('EPSG:4326', inplace=True)

# 3. Map season ordinals to labels
season_map = {1: 'Post to Pre Monsoon (Jan-June)', 2: 'Pre to Post Monsoon (July-Dec)'}
model_gdf['season_label'] = model_gdf['season_ordinal'].map(season_map)

# 4. List of variables to plot
variables_to_plot = [
    'Precipitation',
    'ET',
    'RunOff',
    'CWD_Calculated',
    'CWD_GT'

]

# 5. Plot each variable by season
for season in model_gdf['season_label'].dropna().unique():
    season_df = model_gdf[model_gdf['season_label'] == season]

    for var in variables_to_plot:
        if var not in season_df.columns:
            print(f"Skipping {var} — not found in data.")
            continue

        fig, ax = plt.subplots(figsize=(10, 10))

        # Plot India boundaries in the background
        india_state_boundary_map.plot(ax=ax, color='white', edgecolor='black', linewidth=0.5)

        # Plot variable as point heatmap
        season_df.plot(column=var,
                       ax=ax,
                       legend=True,
                       cmap='viridis',
                       markersize=40,
                       alpha=0.7,
                       missing_kwds={'color': 'lightgrey'})

        ax.set_title(f"{var} — {season}", fontsize=16)
        ax.axis('off')
        plt.tight_layout()
        plt.show()


In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Point

# 1. Read India state boundary
india_url = f"{WORKSPACE_ROOT}/19. preprocessing_deep_aquifers/input/Indian_states_boundary.geojson"
india_state_boundary_map = gpd.read_file(india_url)

# 2. Convert model_df to GeoDataFrame
if 'geometry' not in model_df.columns:
    model_gdf = gpd.GeoDataFrame(
        model_df,
        geometry=gpd.points_from_xy(model_df.Longitude, model_df.Latitude),
        crs='EPSG:4326'
    )
else:
    model_gdf = gpd.GeoDataFrame(model_df, geometry='geometry')
    if model_gdf.crs is None:
        model_gdf.set_crs('EPSG:4326', inplace=True)

# 3. Map season ordinals to readable labels
season_map = {1: 'Post to Pre Monsoon (Jan–June)', 2: 'Pre to Post Monsoon (July–Dec)'}
model_gdf['season_label'] = model_gdf['season_ordinal'].map(season_map)

# 4. Covariates to visualize
variables_to_plot = [
    'weighted_avg_yeild',
    'Precipitation'
    'ET',
    'RunOff',
    'DeltaG',
    'G',
    'Precipitation',
    'CWD_Calculated',
    'CWD_GT',

]

# 5. Plot for each variable by season and quartile category
for season in model_gdf['season_label'].dropna().unique():
    season_df = model_gdf[model_gdf['season_label'] == season].copy()

    for var in variables_to_plot:
        if var not in season_df.columns:
            print(f"Skipping {var} — not found in data.")
            continue

        # Drop rows with missing values
        season_var_df = season_df.dropna(subset=[var]).copy()

        # Compute quartile bins
        season_var_df['quartile'] = pd.qcut(season_var_df[var], q=4, labels=['Q1 (0–25%)', 'Q2 (25–50%)', 'Q3 (50–75%)', 'Q4 (75–100%)'])

        # Get consistent colormap bounds
        vmin = season_var_df[var].min()
        vmax = season_var_df[var].max()

        # Set up plot
        fig, axes = plt.subplots(1, 4, figsize=(24, 6), sharex=True, sharey=True)
        quartiles = ['Q1 (0–25%)', 'Q2 (25–50%)', 'Q3 (50–75%)', 'Q4 (75–100%)']

        for i, q in enumerate(quartiles):
            ax = axes[i]
            india_state_boundary_map.plot(ax=ax, color='white', edgecolor='black', linewidth=0.5)

            subset = season_var_df[season_var_df['quartile'] == q]
            subset.plot(
                column=var,
                cmap='viridis',
                ax=ax,
                vmin=vmin,
                vmax=vmax,
                legend=(i == 3),  # Show legend only on last plot
                markersize=30,
                alpha=0.8,
                missing_kwds={'color': 'lightgrey'}
            )
            ax.set_title(f"{q}", fontsize=12)
            ax.axis('off')

        plt.suptitle(f"{var} — {season}", fontsize=18)
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()


In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point

# 1. Read India state boundary
india_url = f"{WORKSPACE_ROOT}/19. preprocessing_deep_aquifers/input/Indian_states_boundary.geojson"
india_state_boundary_map = gpd.read_file(india_url)

# 2. Convert model_df to GeoDataFrame if not already
if 'geometry' not in model_df.columns:
    # Assuming your model_df has 'longitude' and 'latitude' columns
    model_gdf = gpd.GeoDataFrame(
        model_df,
        geometry=gpd.points_from_xy(model_df.Longitude, model_df.Latitude),
        crs='EPSG:4326'
    )
else:
    model_gdf = gpd.GeoDataFrame(model_df, geometry='geometry')
    if model_gdf.crs is None:
        model_gdf.set_crs('EPSG:4326', inplace=True)

# 3. Map season ordinals to labels
season_map = {1: 'Pre-Monsoon', 2: 'Post-Monsoon'}
model_gdf['season_label'] = model_gdf['season_ordinal'].map(season_map)

# 4. List of variables to plot
variables_to_plot = [
    'weighted_avg_yeild',
    'DeltaG',
    'ET',
    'G',
    'Precipitation',
    'CWD_Calculated',
    'CWD_GT',
    'RunOff'
]

# 5. Plot each variable by season
for season in model_gdf['season_label'].dropna().unique():
    season_df = model_gdf[model_gdf['season_label'] == season]

    for var in variables_to_plot:
        if var not in season_df.columns:
            print(f"Skipping {var} — not found in data.")
            continue

        fig, ax = plt.subplots(figsize=(10, 10))

        # Plot India boundaries in the background
        india_state_boundary_map.plot(ax=ax, color='white', edgecolor='black', linewidth=0.5)

        # Plot variable as point heatmap
        season_df.plot(column=var,
                       ax=ax,
                       legend=True,
                       cmap='viridis',
                       markersize=40,
                       alpha=0.7,
                       missing_kwds={'color': 'lightgrey'})

        ax.set_title(f"{var} — {season}", fontsize=16)
        ax.axis('off')
        plt.tight_layout()
        plt.show()


In [ ]:
import geopandas as gpd
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import plotly.colors as pc
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set season label mapping
season_label_map = {
    1: 'Post to Pre Monsoon (Jan–June)',
    2: 'Pre to Post Monsoon (July–Dec)'
}
model_gdf['season_label'] = model_gdf['season_ordinal'].map(season_label_map)

# Filter MWSs with precipitation in [100, 200] for Post Monsoon season
precip_filtered_df = model_gdf[
    (model_gdf['season_ordinal'] == 2) &
    (model_gdf['Precipitation'] >= 100) &
    (model_gdf['Precipitation'] <= 200)
]
random_ids = precip_filtered_df['microwatershed_id'].dropna().unique()
if len(random_ids) < 3:
    print("Not enough valid microwatersheds.")
else:
    selected_ids = np.random.choice(random_ids, size=3, replace=False)
    colors = sns.color_palette("Set2", n_colors=3)
    id_color_map = dict(zip(selected_ids, colors))

    # Plot selected microwatersheds on India map
    fig_map, ax = plt.subplots(figsize=(8, 8))
    india_state_boundary_map.plot(ax=ax, color='white', edgecolor='black', linewidth=0.5)
    for mws_id in selected_ids:
        mws_points = model_gdf[model_gdf['microwatershed_id'] == mws_id]
        mws_points.plot(ax=ax, color=id_color_map[mws_id], label=f"MWS ID: {mws_id}", markersize=40)
    ax.set_title("Selected Microwatersheds (100–200 mm Precipitation)", fontsize=14)
    ax.legend(title="Microwatershed ID")
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    # Prepare data
    selected_df = model_gdf[model_gdf['microwatershed_id'].isin(selected_ids)].copy()
    selected_df = selected_df.sort_values(by=['microwatershed_id', 'year', 'season_ordinal'])
    selected_df['year_season'] = selected_df['year'].astype(str) + "_" + selected_df['season_ordinal'].astype(str)
    selected_df['season_readable'] = selected_df['season_ordinal'].map(season_label_map)

    variables_to_plot = ['Precipitation', 'RunOff', 'ET', 'DeltaG', 'G', 'CWD_Calculated', 'CWD_GT']
    palette = pc.qualitative.Set2
    var_color_map = dict(zip(variables_to_plot, palette[:len(variables_to_plot)]))

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=[
            "Post to Pre Monsoon (Jan–June)",
            "Pre to Post Monsoon (July–Dec)",
            "Combined"
        ]
    )

    trace_labels = []

    for i, mws_id in enumerate(selected_ids):
        mws_df = selected_df[selected_df['microwatershed_id'] == mws_id]

        # Pre Monsoon
        pre_df = mws_df[mws_df['season_ordinal'] == 1]
        for var in variables_to_plot:
            fig.add_trace(
                go.Scatter(
                    x=pre_df['year'],
                    y=pre_df[var],
                    mode='lines+markers',
                    name=var,
                    legendgroup=var,
                    showlegend=(i == 0),
                    marker=dict(color=var_color_map[var]),
                    line=dict(color=var_color_map[var]),
                    hovertemplate=f"<b>{var}</b><br>MWS ID: {mws_id}<br>Season: {season_label_map[1]}<br>Year: %{{x}}<br>Value: %{{y:.2f}}"
                ),
                row=1, col=1
            )
            trace_labels.append((var, mws_id, 'Pre'))

        # Post Monsoon
        post_df = mws_df[mws_df['season_ordinal'] == 2]
        for var in variables_to_plot:
            fig.add_trace(
                go.Scatter(
                    x=post_df['year'],
                    y=post_df[var],
                    mode='lines+markers',
                    name=var,
                    legendgroup=var,
                    showlegend=False,
                    marker=dict(color=var_color_map[var]),
                    line=dict(color=var_color_map[var]),
                    hovertemplate=f"<b>{var}</b><br>MWS ID: {mws_id}<br>Season: {season_label_map[2]}<br>Year: %{{x}}<br>Value: %{{y:.2f}}"
                ),
                row=1, col=2
            )
            trace_labels.append((var, mws_id, 'Post'))

        # Combined
        for var in variables_to_plot:
            fig.add_trace(
                go.Scatter(
                    x=mws_df['year_season'],
                    y=mws_df[var],
                    mode='lines+markers',
                    name=var,
                    legendgroup=var,
                    showlegend=False,
                    marker=dict(color=var_color_map[var]),
                    line=dict(color=var_color_map[var]),
                    hovertemplate=f"<b>{var}</b><br>MWS ID: {mws_id}<br>Season: %{{customdata[0]}}<br>Year_Season: %{{x}}<br>Value: %{{y:.2f}}",
                    customdata=mws_df[['season_label']].values
                ),
                row=1, col=3
            )
            trace_labels.append((var, mws_id, 'Combined'))

    # Variable-level dropdown menu
    buttons = []
    for var in variables_to_plot:
        visibility = [label[0] == var for label in trace_labels]
        buttons.append(dict(
            label=var,
            method="update",
            args=[{"visible": visibility},
                  {"title": f"Microwatershed Time Series — Variable: {var}"}]
        ))

    # Show all button
    buttons.insert(0, dict(
        label="Show All",
        method="update",
        args=[{"visible": [True] * len(trace_labels)},
              {"title": "Microwatershed Time Series — Seasonal View"}]
    ))

    fig.update_layout(
        height=650,
        width=1600,
        title_text="Microwatershed Time Series — Seasonal View",
        legend_title_text='Variable',
        hovermode="x unified",
        updatemenus=[dict(
            type="dropdown",
            direction="down",
            showactive=True,
            buttons=buttons,
            x=1.01,
            y=1.15,
            xanchor='left',
            yanchor='top'
        )]
    )

    fig.update_xaxes(tickangle=45)
    fig.show()


In [ ]:
import geopandas as gpd
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import plotly.colors as pc
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set season label mapping
season_label_map = {
    1: 'Post to Pre Monsoon (Jan–June)',
    2: 'Pre to Post Monsoon (July–Dec)'
}
model_gdf['season_label'] = model_gdf['season_ordinal'].map(season_label_map)

# Filter only season 2 data
season2_df = model_gdf[model_gdf['season_ordinal'] == 2]

# Group by microwatershed and check if all years have precipitation < 200
valid_ids = (
    season2_df.groupby('microwatershed_id')['Precipitation']
    .apply(lambda x: (x < 200).all())
)

# Select only MWS where all season 2 years have Precip < 200
eligible_ids = valid_ids[valid_ids].index.tolist()

if len(random_ids) < 3:
    print("Not enough valid microwatersheds.")
else:
    selected_ids = np.random.choice(eligible_ids, size=3, replace=False)
    colors = sns.color_palette("Set2", n_colors=3)
    id_color_map = dict(zip(selected_ids, colors))

    # Plot selected microwatersheds on India map
    fig_map, ax = plt.subplots(figsize=(8, 8))
    india_state_boundary_map.plot(ax=ax, color='white', edgecolor='black', linewidth=0.5)
    for mws_id in selected_ids:
        mws_points = model_gdf[model_gdf['microwatershed_id'] == mws_id]
        mws_points.plot(ax=ax, color=id_color_map[mws_id], label=f"MWS ID: {mws_id}", markersize=40)
    ax.set_title("Selected Microwatersheds (100–200 mm Precipitation)", fontsize=14)
    ax.legend(title="Microwatershed ID")
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    # Prepare data
    selected_df = model_gdf[model_gdf['microwatershed_id'].isin(selected_ids)].copy()
    selected_df = selected_df.sort_values(by=['microwatershed_id', 'year', 'season_ordinal'])
    selected_df['year_season'] = selected_df['year'].astype(str) + "_" + selected_df['season_ordinal'].astype(str)
    selected_df['season_readable'] = selected_df['season_ordinal'].map(season_label_map)
    # Print selected IDs and precipitation values across time
    print("✅ Selected Microwatershed IDs and their Precipitation by Year/Season:")
    for mws_id in selected_ids:
        mws_precip = selected_df[selected_df['microwatershed_id'] == mws_id][['year', 'season_label', 'Precipitation']]
        print(f"\nMicrowatershed ID: {mws_id}")
        print(mws_precip.to_string(index=False))

    variables_to_plot = ['Precipitation', 'RunOff', 'ET', 'DeltaG', 'G', 'CWD_Calculated', 'CWD_GT']
    palette = pc.qualitative.Set2
    var_color_map = dict(zip(variables_to_plot, palette[:len(variables_to_plot)]))

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=[
            "Post to Pre Monsoon (Jan–June)",
            "Pre to Post Monsoon (July–Dec)",
            "Combined"
        ]
    )

    trace_labels = []

    for i, mws_id in enumerate(selected_ids):
        mws_df = selected_df[selected_df['microwatershed_id'] == mws_id]

        # Pre Monsoon
        pre_df = mws_df[mws_df['season_ordinal'] == 1]
        for var in variables_to_plot:
            fig.add_trace(
                go.Scatter(
                    x=pre_df['year'],
                    y=pre_df[var],
                    mode='lines+markers',
                    name=var,
                    legendgroup=var,
                    showlegend=(i == 0),
                    marker=dict(color=var_color_map[var]),
                    line=dict(color=var_color_map[var]),
                    hovertemplate=f"<b>{var}</b><br>MWS ID: {mws_id}<br>Season: {season_label_map[1]}<br>Year: %{{x}}<br>Value: %{{y:.2f}}"
                ),
                row=1, col=1
            )
            trace_labels.append((var, mws_id, 'Pre'))

        # Post Monsoon
        post_df = mws_df[mws_df['season_ordinal'] == 2]
        for var in variables_to_plot:
            fig.add_trace(
                go.Scatter(
                    x=post_df['year'],
                    y=post_df[var],
                    mode='lines+markers',
                    name=var,
                    legendgroup=var,
                    showlegend=False,
                    marker=dict(color=var_color_map[var]),
                    line=dict(color=var_color_map[var]),
                    hovertemplate=f"<b>{var}</b><br>MWS ID: {mws_id}<br>Season: {season_label_map[2]}<br>Year: %{{x}}<br>Value: %{{y:.2f}}"
                ),
                row=1, col=2
            )
            trace_labels.append((var, mws_id, 'Post'))

        # Combined
        for var in variables_to_plot:
            fig.add_trace(
                go.Scatter(
                    x=mws_df['year_season'],
                    y=mws_df[var],
                    mode='lines+markers',
                    name=var,
                    legendgroup=var,
                    showlegend=False,
                    marker=dict(color=var_color_map[var]),
                    line=dict(color=var_color_map[var]),
                    hovertemplate=f"<b>{var}</b><br>MWS ID: {mws_id}<br>Season: %{{customdata[0]}}<br>Year_Season: %{{x}}<br>Value: %{{y:.2f}}",
                    customdata=mws_df[['season_label']].values
                ),
                row=1, col=3
            )
            trace_labels.append((var, mws_id, 'Combined'))

    # Variable-level dropdown menu
    buttons = []
    for var in variables_to_plot:
        visibility = [label[0] == var for label in trace_labels]
        buttons.append(dict(
            label=var,
            method="update",
            args=[{"visible": visibility},
                  {"title": f"Microwatershed Time Series — Variable: {var}"}]
        ))

    # Show all button
    buttons.insert(0, dict(
        label="Show All",
        method="update",
        args=[{"visible": [True] * len(trace_labels)},
              {"title": "Microwatershed Time Series — Seasonal View"}]
    ))

    fig.update_layout(
        height=650,
        width=1600,
        title_text="Microwatershed Time Series — Seasonal View",
        legend_title_text='Variable',
        hovermode="x unified",
        updatemenus=[dict(
            type="dropdown",
            direction="down",
            showactive=True,
            buttons=buttons,
            x=1.01,
            y=1.15,
            xanchor='left',
            yanchor='top'
        )]
    )

    fig.update_xaxes(tickangle=45)
    fig.show()


In [ ]:
mws_id = '0008000000000000a20a'
year = 2016
season = 2  # Pre to Post Monsoon

# Filter for the specific record
record = model_gdf[
    (model_gdf['microwatershed_id'] == mws_id) &
    (model_gdf['year'] == year) &
    (model_gdf['season_ordinal'] == season)
]

if record.empty:
    print(f"No data found for MWS ID {mws_id} in {year} (July–Dec).")
else:
    row = record.iloc[0]
    print(f"✅ Microwatershed ID: {mws_id} | Year: {year} | Season: Pre to Post Monsoon (July–Dec)")
    print(f"Precipitation      : {row['Precipitation']:.2f}")
    print(f"DeltaG             : {row['DeltaG']:.2f}")
    print(f"RunOff             : {row['RunOff']:.2f}")
    print(f"G                  : {row['G']:.2f}")
    print(f"CWD_Calculated     : {row['CWD_Calculated']:.2f}")
    print(f"CWD_GT             : {row['CWD_GT']:.2f}")
